# SNC — Train the Query-Conditioned Separator

Trains the **scalable** separation model: a U-Net that is told *which*
class to extract and emits a single mask for it. It handles all
ESC-50 + UrbanSound8K + FSD50K classes with one architecture.

Mixtures are generated **on the fly** during training — there is no
multi-gigabyte dataset file.

## Pipeline

| Stage | Script | Output |
|------|--------|--------|
| 0 | (cell) | Download ESC-50 + UrbanSound8K + FSD50K to Drive if missing |
| 1 | `train_conditioned_separator.py` | `separator_unet_film_multi_v2.3.h5` + `_classes.json` |
| 2 | `evaluate_conditioned_separator.py` | SI-SDR results table |
| 3 | `colab_webapp.ipynb` | Launch the Gradio app to remove sounds |

## Before running

**Runtime → Change runtime type → GPU.** Pick the **T4** GPU — newer
GPUs (A100 / L4 / Blackwell) can fail with `CUDA_ERROR_INVALID_PTX`
because the installed TensorFlow has no kernels for them. T4 is fully
supported and trains this model fine.

## 1. Mount Drive and clone the repo

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DRIVE_ROOT = '/content/drive/MyDrive/snc'
GITHUB_USER = 'keremtutumlu'
GITHUB_REPO_NAME = 'selective-noise-cancelling'
BRANCH = 'feature/separator-quality-overhaul'

# Private repo? Add a Colab Secret named GITHUB_TOKEN (key icon, scope: repo).
import os, subprocess, shutil
from pathlib import Path

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if token:
    clone_url = f'https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git'
    print('Using GITHUB_TOKEN from Colab Secrets.')
else:
    clone_url = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git'
    print('No GITHUB_TOKEN secret found — attempting anonymous clone.')

REPO_DIR = Path(f'/content/{GITHUB_REPO_NAME}')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
result = subprocess.run(
    ['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(
        f'git clone failed (exit {result.returncode}). '
        'If the repo is private, add a GITHUB_TOKEN secret in the left sidebar.'
    )
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-5'], check=True)

Using GITHUB_TOKEN from Colab Secrets.
Working dir: /content/selective-noise-cancelling


CompletedProcess(args=['git', 'log', '--oneline', '-5'], returncode=0)

## 2. Symlink data and saved_models to Drive

In [3]:
drive_data = Path(DRIVE_ROOT) / 'data'
drive_models = Path(DRIVE_ROOT) / 'saved_models'
drive_data.mkdir(parents=True, exist_ok=True)
drive_models.mkdir(parents=True, exist_ok=True)

for local, target in [(REPO_DIR / 'data', drive_data),
                       (REPO_DIR / 'saved_models', drive_models)]:
    if local.is_symlink() or local.exists():
        local.unlink() if local.is_symlink() else shutil.rmtree(local)
    local.symlink_to(target)
    print(f'{local} -> {target}')

/content/selective-noise-cancelling/data -> /content/drive/MyDrive/snc/data
/content/selective-noise-cancelling/saved_models -> /content/drive/MyDrive/snc/saved_models


## Stage 0 — Download ESC-50 (only if missing)

In [4]:
import zipfile, urllib.request

archive_dir = REPO_DIR / 'data' / 'raw' / 'archive'
csv_path = archive_dir / 'esc50.csv'
wav_dir = archive_dir / 'audio' / 'audio'

if csv_path.exists() and wav_dir.exists() and len(list(wav_dir.glob('*.wav'))) >= 2000:
    print(f'ESC-50 already present — {len(list(wav_dir.glob("*.wav")))} WAV files.')
else:
    archive_dir.mkdir(parents=True, exist_ok=True)
    zip_path = Path('/content/esc50.zip')
    url = 'https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip'
    print('Downloading ESC-50 (~600 MB on Colab network)...')
    urllib.request.urlretrieve(url, zip_path)

    extract_dir = Path('/content/esc50_extracted')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    src_root = extract_dir / 'ESC-50-master'

    shutil.copy(src_root / 'meta' / 'esc50.csv', csv_path)
    wav_dir.mkdir(parents=True, exist_ok=True)
    print('Copying WAV files to Drive...')
    for wav in (src_root / 'audio').glob('*.wav'):
        shutil.copy(wav, wav_dir / wav.name)

    zip_path.unlink()
    shutil.rmtree(extract_dir)
    print(f'Done — {len(list(wav_dir.glob("*.wav")))} WAV files.')

assert csv_path.exists() and wav_dir.exists()

ESC-50 already present — 2000 WAV files.


In [5]:
# Stage 0b — Download UrbanSound8K (~6 GB, only if missing).
# Adds 10 urban classes; 4 merge into ESC-50 classes, 6 are new.
import tarfile

us8k_dir = REPO_DIR / 'data' / 'raw' / 'urbansound8k'
us8k_csv = us8k_dir / 'metadata' / 'UrbanSound8K.csv'

if us8k_csv.exists():
    print('UrbanSound8K already present.')
else:
    tar_path = Path('/content/urbansound8k.tar.gz')
    url = 'https://zenodo.org/record/1203745/files/UrbanSound8K.tar.gz'
    print('Downloading UrbanSound8K (~6 GB on Colab network)...')
    urllib.request.urlretrieve(url, tar_path)

    print('Extracting...')
    extract_dir = Path('/content/us8k_extract')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    with tarfile.open(tar_path) as t:
        t.extractall(extract_dir)

    # Extracted layout: us8k_extract/UrbanSound8K/{audio,metadata}
    us8k_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(extract_dir / 'UrbanSound8K'), str(us8k_dir))
    tar_path.unlink()
    shutil.rmtree(extract_dir, ignore_errors=True)
    print('UrbanSound8K ready at', us8k_dir)

assert us8k_csv.exists()

UrbanSound8K already present.


In [ ]:
# Stage 0c — Download FSD50K to LOCAL SSD (~30 GB extracted).
# Drive is bypassed for FSD50K specifically:
#   * Drive quota is often the bottleneck (~30 GB extracted)
#   * Drive sync slows training I/O
#   * Python zipfile + zip -FF on ZIP64 archives silently extracts 0 files
#     into Drive-backed paths (the v2.2 failure mode).
# Trained models still go to Drive via the saved_models symlink.
# Cost: each fresh Colab container has to re-download (~24 GB).
import time

LOCAL_FSD = Path('/content/fsd50k_local')
LOCAL_FSD.mkdir(parents=True, exist_ok=True)

# Replace any Drive-backed fsd50k path with a symlink to local SSD.
fsd_link = REPO_DIR / 'data' / 'raw' / 'fsd50k'
fsd_link.parent.mkdir(parents=True, exist_ok=True)
if fsd_link.is_symlink():
    fsd_link.unlink()
elif fsd_link.exists():
    shutil.rmtree(fsd_link)
fsd_link.symlink_to(LOCAL_FSD)
print(f'{fsd_link} -> {LOCAL_FSD}')

fsd_dir = fsd_link
fsd_dev_audio = fsd_dir / 'FSD50K.dev_audio'
fsd_eval_audio = fsd_dir / 'FSD50K.eval_audio'
fsd_dev_csv = fsd_dir / 'FSD50K.ground_truth' / 'dev.csv'
fsd_eval_csv = fsd_dir / 'FSD50K.ground_truth' / 'eval.csv'

base = 'https://zenodo.org/record/4060432/files'


def _fetch(name, max_retries=3):
    """Download ``name`` from Zenodo, resuming if a sized file is already present."""
    out = fsd_dir / name
    if out.exists() and out.stat().st_size > 0:
        print(f'  {name} already downloaded ({out.stat().st_size / 1e9:.2f} GB).')
        return out
    for attempt in range(1, max_retries + 1):
        try:
            print(f'  Downloading {name} (attempt {attempt})...', end=' ', flush=True)
            urllib.request.urlretrieve(f'{base}/{name}', out)
            print(f'{out.stat().st_size / 1e9:.2f} GB.')
            return out
        except Exception as exc:
            print(f'failed: {exc}')
            if out.exists():
                out.unlink()
            if attempt == max_retries:
                raise
            time.sleep(2 ** attempt)
    return out


def _has_audio(audio_dir):
    return audio_dir.is_dir() and any(audio_dir.glob('*.wav'))


def _unsplit_and_extract(part_names, combined_name, target_dir):
    """Fetch every part, unsplit with ``zip -s 0``, extract with ``unzip``.

    Parts and combined zip are kept until extraction is verified, so a
    retry never re-downloads work that already succeeded. Uses ``unzip``
    instead of Python's zipfile module because the latter silently
    drops entries on >4 GB ZIP64 archives (the v2.2 failure mode).
    """
    for part in part_names:
        _fetch(part)
    final = fsd_dir / part_names[-1]
    combined = fsd_dir / combined_name

    if not combined.exists():
        print(f'  Unsplitting into {combined_name} with `zip -s 0`...')
        # `-s 0` = unsplit a multi-volume zip (the official Zenodo command).
        # `-FF` is a damaged-archive recovery tool and produced broken
        # ZIP64 output on the previous run.
        subprocess.run(['zip', '-s', '0', str(final),
                        '--out', str(combined)], check=True)
        print(f'  Combined zip: {combined.stat().st_size / 1e9:.2f} GB')

    print(f'  Extracting {combined_name} with unzip (~10 min for dev)...')
    subprocess.run(['unzip', '-o', '-q', str(combined),
                    '-d', str(fsd_dir)], check=True)
    print(f'  WAVs in {target_dir.name}: '
          f'{sum(1 for _ in target_dir.glob("*.wav"))}')

    if not _has_audio(target_dir):
        raise RuntimeError(
            f'Extraction finished but {target_dir} has no .wav files. '
            f'Parts and combined zip kept at {fsd_dir} so you can inspect '
            f'them; do NOT delete the .z* files before fixing this.'
        )

    # Only delete after extraction is verified.
    for p in part_names:
        (fsd_dir / p).unlink(missing_ok=True)
    combined.unlink(missing_ok=True)


# 1. Ground truth CSVs (~7 MB).
if fsd_dev_csv.exists() and fsd_eval_csv.exists():
    print('FSD50K ground-truth CSVs already present.')
else:
    print('Fetching FSD50K ground truth...')
    gt = _fetch('FSD50K.ground_truth.zip')
    with zipfile.ZipFile(gt) as z:
        z.extractall(fsd_dir)
    gt.unlink()

# 2. Dev audio (~18 GB download, ~30 GB extracted).
if _has_audio(fsd_dev_audio):
    print(f'FSD50K dev audio already present '
          f'({sum(1 for _ in fsd_dev_audio.glob("*.wav"))} .wav files).')
else:
    print('Fetching FSD50K dev audio (6 parts, ~18 GB download)...')
    _unsplit_and_extract(
        ['FSD50K.dev_audio.z01', 'FSD50K.dev_audio.z02',
         'FSD50K.dev_audio.z03', 'FSD50K.dev_audio.z04',
         'FSD50K.dev_audio.z05', 'FSD50K.dev_audio.zip'],
        '_dev_combined.zip', fsd_dev_audio,
    )

# 3. Eval audio (~6 GB download, ~10 GB extracted).
if _has_audio(fsd_eval_audio):
    print(f'FSD50K eval audio already present '
          f'({sum(1 for _ in fsd_eval_audio.glob("*.wav"))} .wav files).')
else:
    print('Fetching FSD50K eval audio (2 parts, ~6 GB download)...')
    _unsplit_and_extract(
        ['FSD50K.eval_audio.z01', 'FSD50K.eval_audio.zip'],
        '_eval_combined.zip', fsd_eval_audio,
    )

assert fsd_dev_csv.exists() and _has_audio(fsd_dev_audio) and _has_audio(fsd_eval_audio)
print(f'FSD50K ready at {fsd_dir} -> {fsd_dir.resolve()}')

## 3. Install dependencies

In [7]:
!pip install -q librosa==0.11.0 soundfile scikit-learn pandas
import tensorflow as tf
print('TF version :', tf.__version__)
print('GPUs       :', tf.config.list_physical_devices('GPU'))

TF version : 2.20.0
GPUs       : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Stage 1 — Train the conditioned separator

Mixtures are generated on the fly — no dataset file. On a T4 this is
roughly 45-90 minutes depending on `epochs` / `steps_per_epoch`. The
best checkpoint is written to Drive as it improves.

In [8]:
!python src/model_training/train_conditioned_separator.py

2026-05-25 23:03:00,084 - INFO - ESC-50: 2000 clips, 50 classes.
2026-05-25 23:42:52,985 - INFO - UrbanSound8K: 8732 clips, 10 classes.
2026-05-25 23:42:56,524 - INFO - FSD50K: 0 clips, 0 classes.
2026-05-25 23:42:56,524 - INFO - Merged: 56 classes, 10732 clips total.
2026-05-25 23:42:56,524 - INFO - SeparationMixer ready: 56 classes.
2026-05-25 23:43:09,573 - INFO - ESC-50: 2000 clips, 50 classes.
2026-05-25 23:44:37,256 - INFO - UrbanSound8K: 8732 clips, 10 classes.
2026-05-25 23:44:40,269 - INFO - FSD50K: 0 clips, 0 classes.
2026-05-25 23:44:40,269 - INFO - Merged: 56 classes, 10732 clips total.
2026-05-25 23:44:40,270 - INFO - SeparationMixer ready: 56 classes.
2026-05-25 23:44:40.632058: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1779752680.633504   21181 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/devic

## Stage 2 - Smoke Test

In [9]:
!python src/model_training/diagnose_model.py


Model diagnostic — separator_unet_film_multi_v2.2.h5
Data root    — /content/selective-noise-cancelling/data/raw

2026-05-26 01:06:39.121893: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1779757599.123431   57774 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
2026-05-26 01:06:55,956 - INFO - ESC-50: 2000 clips, 50 classes.
2026-05-26 01:08:26,207 - INFO - UrbanSound8K: 8732 clips, 10 classes.
2026-05-26 01:08:29,611 - INFO - FSD50K: 0 clips, 0 classes.
2026-05-26 01:08:29,612 - INFO - Merged: 56 classes, 10732 clips total.
2026-05-26 01:08:29,612 - INFO - SeparationMixer ready: 56 classes.
Test 1 — Output magnitude on a known present class
  Class                   Input max   Output max   Out/In
  

## Stage 3 — Evaluate (SI-SDR)

In [10]:
!python src/model_training/evaluate_conditioned_separator.py


Conditioned Separator — Evaluation Report
Model : /content/selective-noise-cancelling/saved_models/separation_models/separator_unet_film_multi_v2.2.h5

2026-05-26 01:08:59,432 - INFO - ESC-50: 2000 clips, 50 classes.
2026-05-26 01:10:29,315 - INFO - UrbanSound8K: 8732 clips, 10 classes.
2026-05-26 01:10:32,710 - INFO - FSD50K: 0 clips, 0 classes.
2026-05-26 01:10:32,711 - INFO - Merged: 56 classes, 10732 clips total.
2026-05-26 01:10:32,711 - INFO - SeparationMixer ready: 56 classes.
2026-05-26 01:10:33.727335: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1779757833.728838   59799 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Running separation on 800 test mixtures... 2026-05-26 01:10:47.032677: E 

## Stage 4 - Detection — precision / recall on synthetic multi-class mixtures

In [11]:
!python src/model_training/evaluate_detection.py


Detection Evaluation — separator_unet_film_multi_v2.2.h5
Test mixtures: 200

2026-05-26 01:10:59.969932: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1779757859.971439   60523 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
2026-05-26 01:11:16,180 - INFO - ESC-50: 2000 clips, 50 classes.
2026-05-26 01:12:45,190 - INFO - UrbanSound8K: 8732 clips, 10 classes.
2026-05-26 01:12:48,239 - INFO - FSD50K: 0 clips, 0 classes.
2026-05-26 01:12:48,240 - INFO - Merged: 56 classes, 10732 clips total.
2026-05-26 01:12:48,240 - INFO - SeparationMixer ready: 56 classes.
2026-05-26 01:12:51.346497: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal ac

## Stage 5 — Remove sounds via the web app

Once training and evaluation are done, run
`notebooks/colab_webapp.ipynb` to launch the Gradio web app. Upload
any audio or video file, tick the classes to remove, and download the
cleaned result.